In [2]:
import os
import sys

# from google.colab import drive
# drive.mount('/content/drive')
# base_path = "/content/drive/MyDrive/Colab Notebooks/Quant"
# os.chdir(base_path)

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.callbacks import LambdaCallback, EarlyStopping, ModelCheckpoint

import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import regularizers
from tensorflow.keras.layers import Input, Dropout, Dense, Layer, Embedding, Lambda
from keras_hub.layers import PositionEmbedding
from tensorflow.keras.layers import Embedding, MultiHeadAttention, LayerNormalization, GlobalMaxPooling1D, GlobalAveragePooling1D, TextVectorization, BatchNormalization
from tensorflow.keras.models import Model, Sequential

import yfinance as yf
from config import config

from dataclasses import dataclass
import glob
from pprint import pprint

In [4]:
config.NUM_HEAD

4

In [4]:
device_name = tf.test.gpu_device_name()
print(f"Using : {device_name}")

Using : 


### Prepare Dataset

In [77]:
def custom_standardization(input_data):
    lowercase = tf.strings.lower(input_data)
    stripped_html = tf.strings.regex_replace(lowercase, "<br />", " ")
    return tf.strings.regex_replace(
        stripped_html, "[%s]" % re.escape("!#$%&'()*+,-./:;<=>?@\\^_`{|}~"), ""
    )

def get_vectorize_layer(texts, vocab_size, max_seq, special_tokens=["[MASK]"]):
    vectorize_layer = TextVectorization(
        max_tokens = vocab_size,
        output_mode = "int",
        standardize = custom_standardization,
        output_sequence_length = max_seq
    )
    vectorize_layer.adapt(texts)

    vocab = vectorize_layer.get_vocabulary()
    vocab = vocab[2: vocab_size - len(special_tokens)] + ["[mask]"]
    vectorize_layer.set_vocabulary(vocab)
    return vectorize_layer

def encode(texts):
    encoded_texts = vectorize_layer(texts)
    return encoded_texts.numpy()

def get_masked_input_and_labels(encoded_texts):
    inp_mask = np.random.rand(*encoded_texts.shape) < 0.15
    inp_mask[encoded_texts <= 2] = False
    labels = -1 * np.ones(encoded_texts.shape, dtype = int)
    labels[inp_mask] = encoded_texts[inp_mask]

    encoded_texts_masked = np.copy(encoded_texts)
    inp_mask_2mask = inp_mask & (np.random.rand(*encoded_texts.shape) < 0.90)
    encoded_texts_masked[inp_mask_2mask] = (mask_token_id)

    inp_mask_2random = inp_mask_2mask & (np.random.rand(*encoded_texts.shape) < 1/9)
    encoded_texts_masked[inp_mask_2random] = np.random.randint(3, mask_token_id, inp_mask_2random.sum())

    sample_weights = np.ones(labels.shape)
    sample_weights[labels == -1] = 0

    y_labels = np.copy(encoded_texts)

    return encoded_texts_masked, y_labels, sample_weights

In [78]:
def custom_standardization(input_data):
    lowercase = tf.strings.lower(input_data)
    stripped_html = tf.strings.regex_replace(lowercase, "<br />", " ")
    return tf.strings.regex_replace(
        stripped_html, "[%s]" % re.escape("!#$%&'()*+,-./:;<=>?@\\^_`{|}~"), ""
    )

def get_vectorize_layer(texts, vocab_size, max_seq, special_tokens=["[MASK]"]):
    vectorize_layer = TextVectorization(
        max_tokens = vocab_size,
        output_mode = "int",
        standardize = custom_standardization,
        output_sequence_length = max_seq
    )
    vectorize_layer.adapt(texts)

    vocab = vectorize_layer.get_vocabulary()
    vocab = vocab[2: vocab_size - len(special_tokens)] + ["[mask]"]
    vectorize_layer.set_vocabulary(vocab)
    return vectorize_layer

def encode(texts):
    encoded_texts = vectorize_layer(texts)
    return encoded_texts.numpy()

def get_masked_input_and_labels(encoded_texts):
    inp_mask = np.random.rand(*encoded_texts.shape) < 0.15
    inp_mask[encoded_texts <= 2] = False
    labels = -1 * np.ones(encoded_texts.shape, dtype = int)
    labels[inp_mask] = encoded_texts[inp_mask]

    encoded_texts_masked = np.copy(encoded_texts)
    inp_mask_2mask = inp_mask & (np.random.rand(*encoded_texts.shape) < 0.90)
    encoded_texts_masked[inp_mask_2mask] = (mask_token_id)

    inp_mask_2random = inp_mask_2mask & (np.random.rand(*encoded_texts.shape) < 1/9)
    encoded_texts_masked[inp_mask_2random] = np.random.randint(3, mask_token_id, inp_mask_2random.sum())

    sample_weights = np.ones(labels.shape)
    sample_weights[labels == -1] = 0

    y_labels = np.copy(encoded_texts)

    return encoded_texts_masked, y_labels, sample_weights

In [50]:
news_raw = pd.read_csv(os.path.join("data", "abcnews-date-text.csv"))
news_text_raw = news_raw["headline_text"]

In [51]:
# For Local Machine: 

vectorize_layer = get_vectorize_layer(
    news_text_raw.tolist(),
    config.VOCAB_SIZE,
    config.MAX_LEN,
    special_tokens=["[mask]"],
)

mask_token_id = vectorize_layer(["[mask]"]).numpy()[0][0]

x_all_encoded = encode(news_text_raw)

x_masked_train, y_masked_labels, sample_weights = get_masked_input_and_labels(x_all_encoded)

mlm_ds = tf.data.Dataset.from_tensor_slices(
    (x_masked_train, y_masked_labels, sample_weights)
)
mlm_ds = mlm_ds.shuffle(1000).batch(config.BATCH_SIZE)
mlm_ds_small = mlm_ds.shard(num_shards=256, index=0)

In [8]:
# For Google Colab:
# vectorize_layer = get_vectorize_layer(
#     news_text_raw.tolist(),
#     config.VOCAB_SIZE,
#     config.MAX_LEN,
#     special_tokens=["[mask]"],
# )
# mask_token_id = vectorize_layer(["[mask]"]).numpy()[0][0]

# # mlm_ds was created on a local machine, then added to the drive
# mlm_ds = tf.data.Dataset.load(os.path.join("data", "mlm_dataset_final"))
# mlm_ds_small = mlm_ds.shard(num_shards=12, index=0)

In [52]:
sentiment_raw = pd.read_csv(os.path.join("data", "sentiment.csv"), encoding='latin1', header = None)
sentiment_raw.columns = ["Output", "Input"]
sentiment_dataset = sentiment_raw[["Input", "Output"]]

df_0 = sentiment_dataset[sentiment_dataset["Output"] == "negative"]
df_1 = sentiment_dataset[sentiment_dataset["Output"] == "neutral"]
df_2 = sentiment_dataset[sentiment_dataset["Output"] == "positive"]

min_label = min(len(df_0), len(df_1), len(df_2))
df_0_downsampled = resample(df_0, replace=False, n_samples=min_label, random_state=42)
df_1_downsampled = resample(df_1, replace=False, n_samples=min_label, random_state=42)
df_2_downsampled = resample(df_2, replace=False, n_samples=min_label, random_state=42)

sentiment_dataset_balanced = pd.concat([df_0_downsampled, df_1_downsampled, df_2_downsampled])

X, y = sentiment_dataset_balanced['Input'], sentiment_dataset_balanced['Output']
X_encoded = encode(X)

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.1, random_state=42
)

train_classifier_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1000).batch(config.BATCH_SIZE))
test_classifier_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(config.BATCH_SIZE)

In [53]:
print(train_classifier_ds)

<_BatchDataset element_spec=(TensorSpec(shape=(None, 128), dtype=tf.int64, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


In [88]:
print(mlm_ds.element_spec)

(TensorSpec(shape=(None, 128), dtype=tf.int64, name=None), TensorSpec(shape=(None, 128), dtype=tf.int64, name=None), TensorSpec(shape=(None, 128), dtype=tf.float32, name=None))


### Create MLM Model & BERT

``mlm_ds`` : dataset for MLM

``X_train``, ``y_train``, ``X_test``, ``y_test`` : dataset for Training & Validation

In [ ]:
class MultiHeadSelfAttention(Layer): 
    def __init__(self, embed_dim, num_heads=8): 
        super(MultiHeadSelfAttention, self).__init__() 
        self.supports_masking = True

        self.embed_dim = embed_dim 
        self.num_heads = num_heads 
        self.projection_dim = embed_dim // num_heads 
        self.query_dense = Dense(embed_dim) 
        self.key_dense = Dense(embed_dim) 
        self.value_dense = Dense(embed_dim) 
        self.combine_heads = Dense(embed_dim) 
 

    def attention(self, query, key, value, mask=None):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)

        if mask is not None:
            scaled_score += (mask * -1e9)  

        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value) 
        return output, weights


    def split_heads(self, x, batch_size): 
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim)) 
        return tf.transpose(x, perm=[0, 2, 1, 3]) 


    def call(self, inputs, mask = None): 
        batch_size = tf.shape(inputs)[0] 
        query = self.query_dense(inputs) 
        key = self.key_dense(inputs) 
        value = self.value_dense(inputs) 
        query = self.split_heads(query, batch_size) 
        key = self.split_heads(key, batch_size) 
        value = self.split_heads(value, batch_size) 
        attention, _ = self.attention(query, key, value, mask) 
        attention = tf.transpose(attention, perm=[0, 2, 1, 3]) 
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim)) 
        output = self.combine_heads(concat_attention) 
        return output 

class TransformerBlock(Layer): 
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1): 
        super(TransformerBlock, self).__init__() 
        self.supports_masking = True

        self.att = MultiHeadSelfAttention(embed_dim, num_heads) 
        self.ffn = tf.keras.Sequential([ 
            Dense(ff_dim, activation="relu"), 
            Dense(embed_dim), 
        ]) 

        self.layernorm1 = LayerNormalization(epsilon=1e-6) 
        self.layernorm2 = LayerNormalization(epsilon=1e-6) 
        self.dropout1 = Dropout(rate) 
        self.dropout2 = Dropout(rate) 
 

    def call(self, inputs, training, mask=None): 
        attn_output = self.att(inputs, mask=mask) 
        attn_output = self.dropout1(attn_output, training=training) 
        out1 = self.layernorm1(inputs + attn_output) 
        ffn_output = self.ffn(out1) 
        ffn_output = self.dropout2(ffn_output, training=training) 
        return self.layernorm2(out1 + ffn_output) 

class TransformerEncoder(Layer): 
    def __init__(self, num_layers, embed_dim, num_heads, ff_dim, rate=0.1): 
        super(TransformerEncoder, self).__init__() 
        self.supports_masking = True

        self.num_layers = num_layers 
        self.embed_dim = embed_dim 
        self.enc_layers = [TransformerBlock(embed_dim, num_heads, ff_dim, rate) for _ in range(num_layers)] 
        self.dropout = Dropout(rate) 

    def call(self, inputs, training=False, mask=None, input_ids=None): 
        x = inputs
        if mask is None:
            if input_ids is None:
                raise ValueError("input_ids must be provided to create padding mask")
            mask = tf.cast(tf.math.not_equal(input_ids, 0), tf.float32)
            mask = mask[:, tf.newaxis, tf.newaxis, :]  # [batch, 1, 1, seq_len]
            
        for layer in self.enc_layers:
            x = layer(x, training=training, mask=mask)
        return x 

In [ ]:
class MultiHeadSelfAttention(Layer): 
    def __init__(self, embed_dim, num_heads=8): 
        super(MultiHeadSelfAttention, self).__init__() 
        self.embed_dim = embed_dim 
        self.num_heads = num_heads 
        self.projection_dim = embed_dim // num_heads 
        self.query_dense = Dense(embed_dim) 
        self.key_dense = Dense(embed_dim) 
        self.value_dense = Dense(embed_dim) 
        self.combine_heads = Dense(embed_dim) 
 

    def attention(self, query, key, value, mask=None): 
        score = tf.matmul(query, key, transpose_b=True) 
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32) 
        scaled_score = score / tf.math.sqrt(dim_key) 

        if mask is not None:
            scaled_score += (mask * -1e9)

        weights = tf.nn.softmax(scaled_score, axis=-1) 
        output = tf.matmul(weights, value) 
        return output, weights 


    def split_heads(self, x, batch_size): 
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim)) 
        return tf.transpose(x, perm=[0, 2, 1, 3]) 


    def call(self, inputs, mask=None): 
        batch_size = tf.shape(inputs)[0] 
        query = self.query_dense(inputs) 
        key = self.key_dense(inputs) 
        value = self.value_dense(inputs) 
        query = self.split_heads(query, batch_size) 
        key = self.split_heads(key, batch_size) 
        value = self.split_heads(value, batch_size) 
        attention, _ = self.attention(query, key, value) 
        attention = tf.transpose(attention, perm=[0, 2, 1, 3]) 
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim)) 
        output = self.combine_heads(concat_attention) 
        return output 

class TransformerBlock(Layer): 
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1): 
        super(TransformerBlock, self).__init__() 
        self.att = MultiHeadSelfAttention(embed_dim, num_heads) 
        self.ffn = tf.keras.Sequential([ 
            Dense(ff_dim, activation="relu"), 
            Dense(embed_dim), 
        ]) 

        self.layernorm1 = LayerNormalization(epsilon=1e-6) 
        self.layernorm2 = LayerNormalization(epsilon=1e-6) 
        self.dropout1 = Dropout(rate) 
        self.dropout2 = Dropout(rate) 
 

    def call(self, inputs, training, mask=None): 
        attn_output = self.att(inputs) 
        attn_output = self.dropout1(attn_output, training=training) 
        out1 = self.layernorm1(inputs + attn_output) 
        ffn_output = self.ffn(out1) 
        ffn_output = self.dropout2(ffn_output, training=training) 
        return self.layernorm2(out1 + ffn_output) 

class TransformerEncoder(Layer): 
    def __init__(self, num_layers, embed_dim, num_heads, ff_dim, rate=0.1): 
        super(TransformerEncoder, self).__init__() 
        self.num_layers = num_layers 
        self.embed_dim = embed_dim 
        self.enc_layers = [TransformerBlock(embed_dim, num_heads, ff_dim, rate) for _ in range(num_layers)] 
        self.dropout = Dropout(rate) 

    def call(self, inputs, training=False, mask=None, input_ids=None): 
        x = inputs 

        if mask is None:
            if input_ids is None:
                raise ValueError("input_ids must be provided to create padding mask")
            mask = tf.cast(tf.math.not_equal(input_ids, 0), tf.float32)
            mask = mask[:, tf.newaxis, tf.newaxis, :]

        for i in range(self.num_layers): 
            x = self.enc_layers[i](x, training=training, mask=mask) 
        return x 

In [ ]:
# class MultiHeadSelfAttention(Layer): 
#     def __init__(self, embed_dim, num_heads=8): 
#         super(MultiHeadSelfAttention, self).__init__() 
#         self.embed_dim = embed_dim 
#         self.num_heads = num_heads 
#         self.projection_dim = embed_dim // num_heads 
#         self.query_dense = Dense(embed_dim) 
#         self.key_dense = Dense(embed_dim) 
#         self.value_dense = Dense(embed_dim) 
#         self.combine_heads = Dense(embed_dim) 
 

#     def attention(self, query, key, value): 
#         score = tf.matmul(query, key, transpose_b=True) 
#         dim_key = tf.cast(tf.shape(key)[-1], tf.float32) 
#         scaled_score = score / tf.math.sqrt(dim_key) 
#         weights = tf.nn.softmax(scaled_score, axis=-1) 
#         output = tf.matmul(weights, value) 
#         return output, weights 


#     def split_heads(self, x, batch_size): 
#         x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim)) 
#         return tf.transpose(x, perm=[0, 2, 1, 3]) 


#     def call(self, inputs): 
#         batch_size = tf.shape(inputs)[0] 
#         query = self.query_dense(inputs) 
#         key = self.key_dense(inputs) 
#         value = self.value_dense(inputs) 
#         query = self.split_heads(query, batch_size) 
#         key = self.split_heads(key, batch_size) 
#         value = self.split_heads(value, batch_size) 
#         attention, _ = self.attention(query, key, value) 
#         attention = tf.transpose(attention, perm=[0, 2, 1, 3]) 
#         concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim)) 
#         output = self.combine_heads(concat_attention) 
#         return output 

# class TransformerBlock(Layer): 
#     def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1): 
#         super(TransformerBlock, self).__init__() 
#         self.att = MultiHeadSelfAttention(embed_dim, num_heads) 
#         self.ffn = tf.keras.Sequential([ 
#             Dense(ff_dim, activation="relu"), 
#             Dense(embed_dim), 
#         ]) 

#         self.layernorm1 = LayerNormalization(epsilon=1e-6) 
#         self.layernorm2 = LayerNormalization(epsilon=1e-6) 
#         self.dropout1 = Dropout(rate) 
#         self.dropout2 = Dropout(rate) 
 

#     def call(self, inputs, training): 
#         attn_output = self.att(inputs) 
#         attn_output = self.dropout1(attn_output, training=training) 
#         out1 = self.layernorm1(inputs + attn_output) 
#         ffn_output = self.ffn(out1) 
#         ffn_output = self.dropout2(ffn_output, training=training) 
#         return self.layernorm2(out1 + ffn_output) 

# class TransformerEncoder(Layer): 
#     def __init__(self, num_layers, embed_dim, num_heads, ff_dim, rate=0.1): 
#         super(TransformerEncoder, self).__init__() 
#         self.num_layers = num_layers 
#         self.embed_dim = embed_dim 
#         self.enc_layers = [TransformerBlock(embed_dim, num_heads, ff_dim, rate) for _ in range(num_layers)] 
#         self.dropout = Dropout(rate) 

#     def call(self, inputs, training=False): 
#         x = inputs 
#         for i in range(self.num_layers): 
#             x = self.enc_layers[i](x, training=training) 
#         return x 

In [66]:
# Class for MLM
loss_fn = keras.losses.SparseCategoricalCrossentropy(
    from_logits=False, reduction=tf.keras.losses.Reduction.NONE
)
loss_tracker = keras.metrics.Mean(name="loss")

class MaskedLanguageModel(keras.Model):
    def compute_loss(self, x=None, y=None, y_pred=None, sample_weight=None):
        # loss shape: [batch, seq_len]
        loss = loss_fn(y, y_pred)
        if sample_weight is not None:
            sample_weight = tf.cast(sample_weight, loss.dtype)
            loss = loss * sample_weight  # [batch, seq_len] element-wise
        # mean only over unmasked tokens
        mean_loss = tf.reduce_sum(loss) / (tf.reduce_sum(sample_weight) + 1e-8)
        return mean_loss

    def compute_metrics(self, x, y, y_pred, sample_weight):
        return {"loss": loss_tracker.result()}

    @property
    def metrics(self):
        return [loss_tracker]

In [67]:
def create_masked_language_bert_model():
    input_layer = Input(shape=(config.MAX_LEN,), dtype="int64")

    word_embeddings = Embedding(
        input_dim=config.VOCAB_SIZE, 
        output_dim=config.EMBED_DIM, 
        mask_zero = False
    )(input_layer)

    position_embeddings = PositionEmbedding(
        sequence_length=config.MAX_LEN
    )(input_layer)

    embeddings = word_embeddings + position_embeddings

    encoder_layer = TransformerEncoder(config.NUM_LAYERS, config.EMBED_DIM, config.NUM_HEAD, config.FF_DIM)
    encoder_output = encoder_layer(embeddings, input_ids=input_layer, training=True)
    
    mlm_output = Dense(config.VOCAB_SIZE, name="mlm_cls", activation="softmax")(encoder_output)
    
    mlm_model = MaskedLanguageModel(input_layer, mlm_output, name="masked_bert_model")

    optimizer = keras.optimizers.Adam(learning_rate=config.LR)
    mlm_model.compile(optimizer=optimizer)
    return mlm_model

### Train MLM Model

In [68]:
class MaskedTextGenerator(keras.callbacks.Callback):
    def __init__(self, sample_tokens, top_k=5):
        self.sample_tokens = sample_tokens
        self.k = top_k

    def decode(self, tokens):
        return " ".join([
            id2token.get(int(t), "[UNK]")
            for t in tokens if t != 0
        ])

    def convert_ids_to_tokens(self, id):
        return id2token[id]

    def on_epoch_end(self, epoch, logs=None):
        prediction = self.model.predict(self.sample_tokens)

        masked_index = np.where(self.sample_tokens == mask_token_id)
        masked_index = masked_index[1]
        mask_prediction = prediction[0][masked_index]

        top_indices = mask_prediction[0].argsort()[-self.k :][::-1]
        values = mask_prediction[0][top_indices]

        for i in range(len(top_indices)):
            p = top_indices[i]
            v = values[i]
            tokens = np.copy(sample_tokens[0])
            tokens[masked_index[0]] = p
            result = {
                "input_text": self.decode(sample_tokens[0].numpy()),
                "prediction": self.decode(tokens),
                "probability": v,
                "predicted mask token": self.convert_ids_to_tokens(p),
            }
            pprint(result)

def save_model_weights(model, file_name, folder_name):
    os.makedirs(folder_name, exist_ok=True)

    words = file_name.split(".")

    model_name = words[0]

    existing_files = [f for f in os.listdir(folder_name) if f.startswith(model_name)]
    next_number = len(existing_files) + 1
    words[0] = f"{model_name}_{next_number}"

    file_name = ".".join(words)
    save_path = os.path.join(folder_name, file_name)

    model.save_weights(save_path)
    print(f"model saved to: {save_path}")

In [69]:
id2token = dict(enumerate(vectorize_layer.get_vocabulary()))
token2id = {y: x for x, y in id2token.items()}

sample_tokens = vectorize_layer(["I have watched this [mask] and it was awesome"])
generator_callback = MaskedTextGenerator(sample_tokens.numpy())
checkpoint_callback = ModelCheckpoint(
    os.path.join("models", "model_weights_cp_best.weights.h5"), 
    save_best_only=True,
    monitor="loss",
    save_weights_only=True,
    mode="min",
    verbose=1
)


In [70]:
sample_tokens = vectorize_layer(["I have watched this [mask] and it was awesome"])
generator_callback = MaskedTextGenerator(sample_tokens.numpy())

bert_masked_model = create_masked_language_bert_model()
bert_masked_model.summary()

Model: "masked_bert_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_18      │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_8         │ (None, 128, 128)  │  3,840,000 │ input_layer_18[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ position_embedding… │ (None, 128)       │     16,384 │ input_layer_18[0… │
│ (PositionEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 128, 128)  │          0 │ embedding_8[0][0… │
│                     │                   │            │ position_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, 128, 128)  │    396,544 │ add_8[0][0],      │
│ (TransformerEncode… │                   │            │ input_layer_18[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlm_cls (Dense)     │ (None, 128,       │  3,870,000 │ transformer_enco… │
│                     │ 30000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,122,928 (30.99 MB)

 Trainable params: 8,122,928 (30.99 MB)

 Non-trainable params: 0 (0.00 B)

In [71]:
print("model vocab size:", bert_masked_model.output_shape[-1])
print("id2token size:", id2token[2])

model vocab size: 30000
id2token size: to


In [72]:
earlyStopping_callback = EarlyStopping(
    monitor="loss",
    min_delta=0.005,
    patience=5,
    verbose=0,
    mode="auto",
    baseline=None,
    restore_best_weights=True,
)

In [73]:
bert_masked_model.fit(mlm_ds_small, epochs=50, callbacks=[generator_callback, checkpoint_callback, earlyStopping_callback])
save_model_weights(bert_masked_model, "bert_masked_model.weights.h5", "models")

Epoch 1/50


2026-02-28 00:58:51.480351: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: INVALID_ARGUMENT: Incompatible shapes: [16,128,128] vs. [16,128]
	 [[{{function_node __inference_one_step_on_data_68221}}{{node gradient_tape/masked_bert_model_1/Add/BroadcastGradientArgs}}]]


InvalidArgumentError: Graph execution error:

Detected at node gradient_tape/masked_bert_model_1/Add/BroadcastGradientArgs defined at (most recent call last):
  File "/opt/anaconda3/lib/python3.12/runpy.py", line 198, in _run_module_as_main

  File "/opt/anaconda3/lib/python3.12/runpy.py", line 88, in _run_code

  File "/Users/juhyeongpang/Desktop/Projects/Quant/quant_fresh/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/Users/juhyeongpang/Desktop/Projects/Quant/quant_fresh/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/Users/juhyeongpang/Desktop/Projects/Quant/quant_fresh/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start

  File "/Users/juhyeongpang/Desktop/Projects/Quant/quant_fresh/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/opt/anaconda3/lib/python3.12/asyncio/base_events.py", line 645, in run_forever

  File "/opt/anaconda3/

In [ ]:
bert_masked_model.load_weights("models/bert_masked_model_1.weights.h5")

In [ ]:
print(bert_masked_model)

<MaskedLanguageModel name=masked_bert_model, built=True>


In [ ]:
pretrained_bert_model = keras.Model(
    bert_masked_model.input, bert_masked_model.get_layer("transformer_encoder_1").output
)

In [ ]:
pretrained_bert_model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 256, 128)  │  3,840,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ position_embedding… │ (None, 256, 128)  │     32,768 │ embedding_1[0][0] │
│ (PositionEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 256, 128)  │          0 │ embedding_1[0][0… │
│                     │                   │            │ position_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, 256, 128)  │    793,088 │ add_1[0][0]       │
│ (TransformerEncode… │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,665,856 (17.80 MB)

 Trainable params: 4,665,856 (17.80 MB)

 Non-trainable params: 0 (0.00 B)

In [32]:
class DiNOLikeCentering(tf.keras.layers.Layer):
    def __init__(self, momentum=0.9, **kwargs):
        super().__init__(**kwargs)
        self.momentum = momentum
        self.registered_mean = None

    def build(self, input_shape):
        self.registered_mean = self.add_weight(
            shape=(input_shape[-1],),
            initializer="zeros",
            trainable=False,
            name="centering_mean"
        )
        super().build(input_shape)

    def call(self, x, training=False):
        batch_mean = tf.reduce_mean(x, axis=0)  # feature-wise 평균
        if training:
            self.registered_mean.assign(self.momentum * self.registered_mean + (1 - self.momentum) * batch_mean)
        # centering
        return x - self.registered_mean
    

def create_dino_style_bert_model(sharpen_temp=0.5):
    inputs = Input(shape=(config.MAX_LEN,), dtype="int64")
    sequence_output = pretrained_bert_model(inputs)
    cls_token = sequence_output[:, 0, :]  # [CLS] 토큰

    hidden_layer = Dense(64, activation="relu")(cls_token)
    hidden_layer = BatchNormalization()(hidden_layer)
    hidden_layer = Dropout(0.2)(hidden_layer)
    hidden_layer = Dense(32, activation="relu")(hidden_layer)
    hidden_layer = BatchNormalization()(hidden_layer)
    hidden_layer = Dropout(0.2)(hidden_layer)
    hidden_layer = Dense(16, activation="relu")(hidden_layer)
    hidden_layer = BatchNormalization()(hidden_layer)
    hidden_layer = Dropout(0.2)(hidden_layer)

    # DiNO 스타일 centering (moving average)
    hidden_layer = DiNOLikeCentering(momentum=0.9)(hidden_layer)

    logits = Dense(3)(hidden_layer)

    # sharpening
    outputs = Lambda(lambda x: tf.nn.softmax(x / sharpen_temp))(logits)

    model = Model(inputs, outputs, name="dino_style_classification")

    loss_fn = keras.losses.SparseCategoricalCrossentropy(reduction=None, ignore_class=0)
    optimizer = keras.optimizers.Adam(learning_rate=0.0001)
    model.compile(optimizer=optimizer, loss=loss_fn, metrics=["accuracy"])
    return model

In [ ]:
def create_classifier_bert_model():
    inputs = Input(shape=(config.MAX_LEN,), dtype="int64")
    sequence_output = pretrained_bert_model(inputs)
    cls_token = sequence_output[:, 0, :] 
    hidden_layer = Dense(64, activation="relu")(cls_token)
    hidden_layer = BatchNormalization()(hidden_layer)
    hidden_layer = Dropout(0.2)(hidden_layer)
    hidden_layer = Dense(32, activation="relu")(hidden_layer)
    hidden_layer = BatchNormalization()(hidden_layer)
    hidden_layer = Dropout(0.2)(hidden_layer)
    hidden_layer = Dense(16, activation="relu")(hidden_layer)
    hidden_layer = BatchNormalization()(hidden_layer)
    hidden_layer = Dropout(0.2)(hidden_layer)
    outputs = Dense(3, activation="softmax")(hidden_layer)
    classifer_model = Model(inputs, outputs, name="classification")
    
    loss_fn = keras.losses.SparseCategoricalCrossentropy(reduction=None, ignore_class=0)
    optimizer = keras.optimizers.Adam(learning_rate=1e-3)
    classifer_model.compile(
        optimizer=optimizer, loss=loss_fn, metrics=["accuracy"]
    )
    return classifer_model

In [34]:
pretrained_bert_model.trainable = False
dino_classifer_model = create_dino_style_bert_model()
dino_classifer_model.summary()

Model: "dino_style_classification"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ functional_4 (Functional)       │ (None, 256, 128)       │     4,665,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ di_no_like_centering            │ (None, 16)             │            16 │
│ (DiNOLikeCentering)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 3)              │            51 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 3)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,677,235 (17.84 MB)

 Trainable params: 11,139 (43.51 KB)

 Non-trainable params: 4,666,096 (17.80 MB)

In [ ]:
pretrained_bert_model.trainable = False
classifer_model = create_classifier_bert_model()
classifer_model.summary()

In [40]:
for layer in pretrained_bert_model.layers[:-4] :
    layer.trainable = True

In [41]:
dino_classifer_model.fit(
    train_classifier_ds,
    epochs=5,
    validation_data=test_classifier_ds,
)
save_model_weights(dino_classifer_model, "dino_classifer_model_freeze.weights.h5", "models")

Epoch 1/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.3373 - loss: 1.3598 - val_accuracy: 0.2418 - val_loss: 0.6510
Epoch 2/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.3519 - loss: 1.2466 - val_accuracy: 0.2418 - val_loss: 0.6569
Epoch 3/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.3329 - loss: 1.3921 - val_accuracy: 0.2418 - val_loss: 0.6552
Epoch 4/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.3217 - loss: 1.2382 - val_accuracy: 0.2418 - val_loss: 0.6244
Epoch 5/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.3389 - loss: 1.2394 - val_accuracy: 0.2418 - val_loss: 0.6243
model saved to: models/dino_classifer_model_freeze_2.weights.h5


In [ ]:
# Train the classifier with frozen BERT stage
classifer_model.fit(
    train_classifier_ds,
    epochs=5,
    validation_data=test_classifier_ds,
)
save_model_weights(classifer_model, "classifer_model_freeze.weights.h5", "models")

In [ ]:
# Unfreeze the BERT model for fine-tuning
pretrained_bert_model.trainable = True
optimizer = keras.optimizers.Adam()
classifer_model.compile(
    optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
classifer_model.fit(
    train_classifier_ds,
    epochs=5,
    validation_data=test_classifier_ds,
)
save_model_weights(classifer_model, "classifer_model.weights.h5", "models")

In [37]:
def predict(text, model):
    text_encoded = encode(text)
    # print("Encoded Text: ", text_encoded)
    
    pred = model(text_encoded, training=False)

    probabilities = tf.nn.softmax(pred, axis=-1).numpy()[0]

    pred_class = np.argmax(probabilities)
    sentiment = le.inverse_transform([pred_class])[0]
    confidence_score = probabilities[pred_class]

    class_names = le.classes_
    all_probs = {class_names[i]: float(probabilities[i]) for i in range(len(class_names))}
    
    return sentiment, confidence_score, all_probs

def predict_and_print(text, model):
    new_texts = [text]
    result, confidence, all_probs = predict(new_texts, model)
    
    print(f"Input Headline: \"{text}\"")
    print(f"Top Prediction: {result} ({confidence * 100:.2f}%)")
    print("-" * 30)
    print("Class Probabilities:")
    
    sorted_probs = sorted(all_probs.items(), key=lambda item: item[1], reverse=True)
    
    for label, prob in sorted_probs:
        bar = "#" * int(prob * 20) 
        print(f" - {label:<12}: {prob * 100:>6.2f}% {bar}")

In [39]:
predict_and_print("This is a positive sign for apple", dino_classifer_model)

Input Headline: "This is a positive sign for apple"
Top Prediction: negative (34.80%)
------------------------------
Class Probabilities:
 - negative    :  34.80% ######
 - neutral     :  32.89% ######
 - positive    :  32.31% ######
